## Loading the data

In [2]:
import pandas as pd
import numpy as np

In [3]:
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
geoloc = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
order_payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')

### Completeness Check (Missing Values)

In [4]:
def missing_values_report(df, table_name):
    total = len(df)                                                 # total number of rows in the data
    missing_values = df.isnull().sum()                              # calculating missing values
    missing_per = ((missing_values/total)*100).round(2)             # calculating missing values in percentage

    report = pd.DataFrame({                                         # storing information in dataframe
        'Column': missing_values.index,
        'Missing Count' : missing_values.values,
        'Missing %' : missing_per.values,
        'Data Type' : df.dtypes.values
    })

    report = report[report['Missing Count'] > 0]                    # isolating the missing values data
    report['Severity'] = report['Missing %'].apply(                 # calculating severity
        lambda x: 'Critical' if x > 20
        else 'Moderate' if x > 5
        else 'Low'
    )
    print(f"\n                      =======Missing Value Report: {table_name} ======= ")
    print(report.to_string(index=False))
    return report


missing_values_report(orders, "orders")


                      =======Missing Value Report: orders ======= 
                       Column  Missing Count  Missing % Data Type Severity
            order_approved_at            160       0.16    object      Low
 order_delivered_carrier_date           1783       1.79    object      Low
order_delivered_customer_date           2965       2.98    object      Low


,Column,Missing Count,Missing %,Data Type,Severity
4,order_approved_at,160,0.16,object,Low
5,order_delivered_carrier_date,1783,1.79,object,Low
6,order_delivered_customer_date,2965,2.98,object,Low


### Duplicates Check

In [5]:
def duplicate_report(df, key_col, table_name):
    total = len(df)
    duplicate_rows = df.duplicated().sum()
    duplicate_keys = df[key_col].duplicated().sum()

    print(f"\n              ======Duplicate Report: {table_name}======")
    print(f"Total Rows              :{total}")
    print(f"Fully Duplicate Rows    :{duplicate_rows} ({round(duplicate_rows*100/total, 2)})")
        


    return duplicate_report

duplicate_report(orders, "order_id", "Orders")


              ======Duplicate Report: Orders======
Total Rows              :99441
Fully Duplicate Rows    :0 (0.0)


<function __main__.duplicate_report(df, key_col, table_name)>

## Validity Check (Format & Range)

In [6]:
def validity_report(df, table_name):
    print(f"\n          ======Validity Report: {table_name} =   ====")

    # date columns - are they parsable?
    date_cols =[col for col in df.columns if'date' in col or 'timestamp' in col]
    
    for col in date_cols:
        try:
            converted_cols = pd.date_time(df[col],errors = 'coerce')
            failed = converted_cols.isnull().sum() - df[col].isnull().sum()
            print(f"Date column '{col}' : {failed} unparseable values")
        except:
            print(f"Date column '{col}' : Could not convert")

    # Numberic columns - check for negatives or outliers

    num_cols = df.select_dtypes(include = [np.number]).columns
    for col in num_cols:
        negatives = (df[col] < 0).sum()
        if negatives > 0:
            print(f"Numeric columns '{col}': {negatives} negative values")
    

validity_report(orders, "Orders")


          ======Validity Report: Orders =   ====
Date column 'order_purchase_timestamp' : Could not convert
Date column 'order_delivered_carrier_date' : Could not convert
Date column 'order_delivered_customer_date' : Could not convert
Date column 'order_estimated_delivery_date' : Could not convert


## Summary Statistics

In [7]:
def summary_stats_report(df, table_name):
    print(f"            ===== Summary Statistics: {table_name}=====")
    print(df.describe(include='all').T)

summary_stats_report(orders, "Orders")

            ===== Summary Statistics: Orders=====
                               count unique                               top  \
order_id                       99441  99441  e481f51cbdc54678b7cc49136f2d6af7   
customer_id                    99441  99441  9ef432eb6251297304e76186b10a928d   
order_status                   99441      8                         delivered   
order_purchase_timestamp       99441  98875               2018-08-02 12:05:26   
order_approved_at              99281  90733               2018-02-27 04:31:10   
order_delivered_carrier_date   97658  81018               2018-05-09 15:48:00   
order_delivered_customer_date  96476  95664               2018-05-08 19:36:48   
order_estimated_delivery_date  99441    459               2017-12-20 00:00:00   

                                freq  
order_id                           1  
customer_id                        1  
order_status                   96478  
order_purchase_timestamp           3  
order_approved_at        

## FULL DATA QUALITY REPORT

In [8]:
def full_data_quality_report(dataframes: dict):
    """
        dataframes: dictionary of {table name: dataframe}
        key_columns: primary key per table
    """

    key_map = {
        'orders':'order_id',
        'customers':'customer_id',
        'products':'product_id',
        'sellers':'seller_id'
    }

    all_missing = []

    for name, df in dataframes.items():
        print(f"\n{'='*50}")
        print(f"TABLE: {name.upper()}")
        print(f"{'='*50}")
        print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")

        m = missing_values_report(df, name)
        all_missing.append(m)

        if name in key_map:
            duplicate_report(df, key_map[name], name)

        validity_report(df, name)

dataframes = {
    'orders' : orders,
    'customers' : customers,
    'products' : products,
    'geoloc' : geoloc, 
    'order_items' : order_items,
    'order_payments' : order_payments,
    'order_reviews' : order_reviews,
    'sellers' : sellers
}

full_data_quality_report(dataframes)




TABLE: ORDERS
Rows: 99441 | Columns: 8

                      =======Missing Value Report: orders ======= 
                       Column  Missing Count  Missing % Data Type Severity
            order_approved_at            160       0.16    object      Low
 order_delivered_carrier_date           1783       1.79    object      Low
order_delivered_customer_date           2965       2.98    object      Low

              ======Duplicate Report: orders======
Total Rows              :99441
Fully Duplicate Rows    :0 (0.0)

          ======Validity Report: orders =   ====
Date column 'order_purchase_timestamp' : Could not convert
Date column 'order_delivered_carrier_date' : Could not convert
Date column 'order_delivered_customer_date' : Could not convert
Date column 'order_estimated_delivery_date' : Could not convert

TABLE: CUSTOMERS
Rows: 99441 | Columns: 5

                      =======Missing Value Report: customers ======= 
Empty DataFrame
Columns: [Column, Missing Count, Missing %, Da